### Search Engine with Tools & Agents with LangChain


In [1]:
## Arxiv -- Reasearch
## Tools Creation


from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [2]:
#created the wrapper for wiki
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=350)
#to run wrapper we need wikipedia query tool
tool_wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
tool_wiki.name

'wikipedia'

In [3]:
#create the arxiv tools and wrapper

api_arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=350)
#To Run wrapper we need arxiv query tool
tool_arxiv = ArxivQueryRun(api_wrapper=api_arxiv_wrapper)

tool_arxiv.name

'arxiv'

In [4]:
#Combine the tools in a list

tools = [tool_wiki,tool_arxiv]

In [5]:
#We can create the custom tools and agents as well [RAG Tool]

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


/Users/apple/Desktop/Langchain/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader("https://docs.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

db = FAISS.from_documents(documents, OpenAIEmbeddings())

retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1396cbb50>, search_kwargs={})

In [7]:
#convert retriever to tool
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(retriever=retriever, name="Langchain Documentation Retriever", description="useful for when you need to answer questions about langchain documentation")

retriever_tool.name

'Langchain Documentation Retriever'

In [8]:
tools = [tool_wiki,tool_arxiv,retriever_tool]

In [9]:
#Run all these tools with an agent and llm
#Combine the tools in a list

from dotenv import load_dotenv
load_dotenv()
import os

groq_api_key = os.getenv("GROQ_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(groq_api_key=groq_api_key, openai_api_key=openai_api_key, model="gpt-4o")

/Users/apple/Desktop/Langchain/venv/lib/python3.11/site-packages/pydantic/main.py:250: UserWarning: WARNING! openai_api_key is not default parameter.
                    openai_api_key was transferred to model_kwargs.
                    Please confirm that openai_api_key is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [10]:
from langchain.agents import create_agent

In [11]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
    # ... (other params)
)
agent = create_agent(model, tools=tools)

In [17]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")


# Use with chat models
messages = [system_msg,HumanMessage(content="What is Langchain? And how do I create a RESTAPI?")]
response = model.invoke(messages)  # Returns AIMessage

In [18]:
response.content

'Short answer first, then concrete steps with code.\n\nWhat is LangChain?\n- LangChain is a Python library that helps you build language-model-powered applications more easily. It provides:\n  - LLM orchestration (chains, prompts, and templates)\n  - Tools and agents to interact with external systems (APIs, databases, calculators, search, etc.)\n  - Memory and retrieval components (short-term/long-term memory, document QA with indexes)\n  - Support for multiple LLM providers (OpenAI, Cohere, HuggingFace, etc.)\n- Use cases: chatbots, document QA, reasoning agents that call tools, data analysis assistants, and more.\n\nExample: a tiny LangChain snippet\n- This shows an LLM-based chain that answers a question using a prompt template.\n\n```python\n# quick-start (requires OpenAI API key)\nfrom langchain.llms import OpenAI\nfrom langchain.prompts import PromptTemplate\nfrom langchain.chains import LLMChain\n\ntemplate = "Question: {question}\\nAnswer:"\nprompt = PromptTemplate(input_variab